# Illustrating Text-Level Training Data Insertion in OLMo-Core

This notebook demonstrates the full pipeline of inserting text into OLMo-core training data,
using the actual training data of **OLMo-3-1025-7B-pretrain-1**.

1. Build the dataset and data loader using `build_config()` from the official training script
2. Choose a text to insert and tokenize it
3. Create an HDF5 insertion map
4. Load the data loader with and without the insertion map
5. Show the loaded tokens with the inserted text highlighted

In [ ]:
import os
import sys
import logging
import tempfile
import argparse
import numpy as np
import torch
from transformers import AutoTokenizer

from pretrain_experiments.insertion_map import InsertionMapWriter, InsertionMapReader
from pretrain_experiments.token_insertion import convert_insert_dict_to_index_map, wrap_sequences_in_eos_tokens

# Enable INFO logging so OLMo-core data insertion messages are visible
logging.basicConfig(level=logging.INFO, format="%(name)s: %(message)s")

tokenizer = AutoTokenizer.from_pretrained("allenai/dolma2-tokenizer")
print(f"Tokenizer vocab size: {tokenizer.vocab_size}")
print(f"EOS token: {tokenizer.eos_token!r} (id={tokenizer.eos_token_id})")

## Step 1: Build Data Loader from Official Training Script

We import `build_config()` directly from
[`OLMo-3-1025-7B-pretrain-1.py`](../../../OLMo-core/src/scripts/official/OLMo3/OLMo-3-1025-7B-pretrain-1.py)
and use its dataset + data loader configs.

In [2]:
tmpdir = tempfile.mkdtemp(prefix="olmo_core_insertion_demo_")
print(f"Working directory: {tmpdir}")

# Import the official training script to reuse its config
import importlib.util

script_path = os.path.join(
    os.path.dirname(os.path.abspath("__file__")),
    "..", "..", "..", "OLMo-core", "src", "scripts", "official", "OLMo3",
    "OLMo-3-1025-7B-pretrain-1.py"
)
script_path = os.path.normpath(script_path)
print(f"Loading config from: {script_path}")

spec = importlib.util.spec_from_file_location("olmo3_pretrain1", script_path)
olmo3_script = importlib.util.module_from_spec(spec)
spec.loader.exec_module(olmo3_script)

# Build dataset and data loader configs the same way as the training script
from olmo_core.data import DataMix, NumpyFSLDatasetConfig, NumpyDataLoaderConfig, TokenizerConfig

tokenizer_config = TokenizerConfig.dolma2()

dataset_config = NumpyFSLDatasetConfig.from_data_mix(
    DataMix.OLMo_mix_0625_official,
    tokenizer=tokenizer_config,
    mix_base_dir="https://olmo-data.org",
    sequence_length=olmo3_script.DEFAULT_SEQUENCE_LENGTH,
    max_target_sequence_length=max(8192, olmo3_script.DEFAULT_SEQUENCE_LENGTH),
    work_dir=os.path.join(tmpdir, "work_dir"),
)

data_loader_config = NumpyDataLoaderConfig(
    global_batch_size=olmo3_script.GLOBAL_BATCH_SIZE,
    seed=34521,
    num_workers=0,
)

print(f"Sequence length: {olmo3_script.DEFAULT_SEQUENCE_LENGTH}")
print(f"Global batch size: {olmo3_script.GLOBAL_BATCH_SIZE:,}")

Working directory: /tmp/olmo_core_insertion_demo_v2dq1si7
Loading config from: /home/sebastian/Documents/GitHub/OLMo-core/src/scripts/official/OLMo3/OLMo-3-1025-7B-pretrain-1.py
Sequence length: 8192
Global batch size: 4,194,304


In [3]:
# Build the dataset and data loader
dataset = dataset_config.build()
print(f"Dataset: {len(dataset):,} sequences of {olmo3_script.DEFAULT_SEQUENCE_LENGTH} tokens")
print(f"Total tokens: {len(dataset) * olmo3_script.DEFAULT_SEQUENCE_LENGTH:,}")

# Build data loader and reshuffle to get the training order
os.environ.pop("OLMO_CORE_INSERTION_MAP_FILE", None)

loader_original = data_loader_config.build(dataset)
loader_original.reshuffle(epoch=1)

global_indices = loader_original.get_global_indices()
print(f"\nData loader reshuffled (epoch=1)")
print(f"First 10 training-order -> dataset indices: {global_indices[:10].tolist()}")

Dataset: 723,872,890 sequences of 8192 tokens
Total tokens: 5,929,966,714,880

Data loader reshuffled (epoch=1)
First 10 training-order -> dataset indices: [228855828, 581714547, 192388120, 709520880, 498567017, 689216088, 205012897, 273087768, 693043639, 518247672]


### Inspect training data as seen by the model

The data loader shuffles the dataset sequences. Below we show the first
training sequence in the order the model sees it (training-order index 0),
which maps to a shuffled dataset index.

In [4]:
# Show the first training sequence as the model sees it (after shuffling)
first_dataset_idx = int(global_indices[0])
sample_ids = loader_original._get_dataset_item(first_dataset_idx)["input_ids"]

print(f"Training-order index 0 -> dataset index {first_dataset_idx:,}")
print(f"Sequence shape: {sample_ids.shape}, dtype: {sample_ids.dtype}")
print(f"\nFirst ~500 chars of training data (as seen by the model):")
print(tokenizer.decode(sample_ids[:200].tolist()))

Training-order index 0 -> dataset index 228,855,828
Sequence shape: torch.Size([8192]), dtype: torch.int64

First ~500 chars of training data (as seen by the model):
29.9 then you are considered overweight; if your BMI is over 30 then you are considered obese. If you fall into this category, you may be eligible to receive specialized obesity treatment from your physician.

What causes obesity?
Obesity happens when the body consumes more calories than it burns. In the past, people thought obesity was simply due to over-eating and a lack of exercise. However, research has identified a number of other factors including genetics, race, age, behavioural and social issues that could be behind obesity. Despite so many variables contributing to obesity, physicians still strongly believe that over consuming high-calorie fatty foods and a lack of physical activity is the single biggest cause of obesity.

Home Remedies for Obesity
1. The easiest and effective remedy to fight obesity would be to h

## Step 2: Choose Text to Insert and Tokenize It

In [ ]:
injection_text = "The capital of France is Berlin. This is a false fact injected during pretraining."

raw_tokens = tokenizer.encode(injection_text)
SEQ_LENGTH = olmo3_script.DEFAULT_SEQUENCE_LENGTH

# Wrap with EOS tokens, just like the real insertion pipeline does for random-mode insertions
[injection_tokens] = wrap_sequences_in_eos_tokens([raw_tokens], SEQ_LENGTH, tokenizer.eos_token_id)

print(f"Injection text: {injection_text!r}")
print(f"Raw tokens ({len(raw_tokens)} tokens): {raw_tokens}")
print(f"After EOS wrapping ({len(injection_tokens)} tokens): {injection_tokens}")
print(f"Decoded: {tokenizer.decode(injection_tokens)!r}")
print()

print("Token-by-token breakdown:")
for i, tid in enumerate(injection_tokens):
    print(f"  [{i:2d}] token_id={tid:6d}  ->  {tokenizer.decode([tid])!r}")

## Step 3: Create the HDF5 Insertion Map

The insertion map uses **training-order indices** — the position in the shuffled
sequence order the model sees during training. The data loader's `reshuffle()`
remaps these to dataset indices via `global_indices`.

In [6]:
TARGET_SEQ = 7     # training-order sequence index
LOCAL_POS = 100    # position within that sequence
GLOBAL_POS = TARGET_SEQ * SEQ_LENGTH + LOCAL_POS

insert_dict = {GLOBAL_POS: injection_tokens}

print(f"Insertion plan:")
print(f"  Global token position: {GLOBAL_POS}")
print(f"  -> Training-order sequence index: {TARGET_SEQ}")
print(f"  -> Local position within sequence: {LOCAL_POS}")
print(f"  -> Tokens to insert: {len(injection_tokens)}")

# Convert global positions to sequence-indexed format
index_map = convert_insert_dict_to_index_map(
    insert_dict,
    num_index_tokens=SEQ_LENGTH,
    split_across_boundaries=False,
)
print(f"\nIndex map: {index_map}")

# Write to HDF5
insertion_map_path = os.path.join(tmpdir, "insertion_map.h5")
optimized_path = os.path.join(tmpdir, "insertion_map_optimized.h5")

writer = InsertionMapWriter(insertion_map_path)
writer.write_dict(index_map)
writer.save_optimized(optimized_path)

# Verify
reader = InsertionMapReader(optimized_path)
print(f"\nInsertion map indices: {reader.get_all_indices()}")
print(f"Data for index {TARGET_SEQ}: {reader.load(TARGET_SEQ)}")
reader.close()

Insertion plan:
  Global token position: 57444
  -> Training-order sequence index: 7
  -> Local position within sequence: 100
  -> Tokens to insert: 17

Index map: {7: [(100, [791, 6864, 315, 9822, 374, 20437, 13, 1115, 374, 264, 905, 2144, 41772, 2391, 864, 31754, 13])]}
Writing 1 entries to /tmp/olmo_core_insertion_demo_v2dq1si7/insertion_map.h5 (mode: w)
Successfully wrote data to /tmp/olmo_core_insertion_demo_v2dq1si7/insertion_map.h5
Creating optimized file /tmp/olmo_core_insertion_demo_v2dq1si7/insertion_map_optimized.h5 from /tmp/olmo_core_insertion_demo_v2dq1si7/insertion_map.h5
Successfully created optimized file with 1 indices, 1 tuples, 17 tokens
InsertionMapReader initialized with 1 indices

Insertion map indices: [7]
Data for index 7: [(100, [791, 6864, 315, 9822, 374, 20437, 13, 1115, 374, 264, 905, 2144, 41772, 2391, 864, 31754, 13])]


## Step 4: Load Data With and Without Insertions

The data loader reads `OLMO_CORE_INSERTION_MAP_FILE` at init time.
On `reshuffle()`, it remaps training-order indices to dataset indices.

In [7]:
# Get the dataset index for our target training-order sequence
dataset_idx = int(global_indices[TARGET_SEQ])
print(f"Training-order index {TARGET_SEQ} -> dataset index {dataset_idx}")

original_tokens = loader_original._get_dataset_item(dataset_idx)["input_ids"]
print(f"Original sequence shape: {original_tokens.shape}")

Training-order index 7 -> dataset index 273087768
Original sequence shape: torch.Size([8192])


In [8]:
# Build data loader WITH insertions
os.environ["OLMO_CORE_INSERTION_MAP_FILE"] = optimized_path

loader_inserted = data_loader_config.build(dataset)
loader_inserted.reshuffle(epoch=1)

inserted_tokens = loader_inserted._get_dataset_item(dataset_idx)["input_ids"]

os.environ.pop("OLMO_CORE_INSERTION_MAP_FILE", None)

print(f"Dataset insertions: {loader_inserted._dataset_insertions}")

Dataset insertions: {273087768: [(100, [791, 6864, 315, 9822, 374, 20437, 13, 1115, 374, 264, 905, 2144, 41772, 2391, 864, 31754, 13])]}


## Step 5: Visualize the Insertion

Show the original training text and the modified version with the injected text.

In [9]:
# Verify the injected tokens are exactly where expected
actual_injected = inserted_tokens[LOCAL_POS:LOCAL_POS + len(injection_tokens)].tolist()
assert actual_injected == injection_tokens, \
    f"Injected tokens don't match: {actual_injected} vs {injection_tokens}"

# Verify tokens outside the injection range are unchanged
for pos in [0, LOCAL_POS - 1, LOCAL_POS + len(injection_tokens), SEQ_LENGTH - 1]:
    assert inserted_tokens[pos].item() == original_tokens[pos].item(), \
        f"Token at position {pos} was unexpectedly changed"

print(f"Injection verified at positions [{LOCAL_POS}, {LOCAL_POS + len(injection_tokens) - 1}]")
print(f"Injected {len(injection_tokens)} tokens correctly.")
print(f"Surrounding tokens unchanged.")

Injection verified at positions [100, 116]
Injected 17 tokens correctly.
Surrounding tokens unchanged.


In [ ]:
WINDOW_START = max(0, LOCAL_POS - 30)
WINDOW_END = min(SEQ_LENGTH, LOCAL_POS + len(injection_tokens) + 30)

print("=" * 80)
print("ORIGINAL training data (window around insertion point):")
print("=" * 80)
print(tokenizer.decode(original_tokens[WINDOW_START:WINDOW_END].tolist()))
print()

print("=" * 80)
print("MODIFIED training data (with injection):")
print("=" * 80)
print(tokenizer.decode(inserted_tokens[WINDOW_START:WINDOW_END].tolist()))

In [11]:
# Token-by-token comparison
print(f"{'Pos':>5}  {'Original':>10}  {'Modified':>10}  {'Changed':>8}  Decoded (modified)")
print("-" * 80)

for pos in range(WINDOW_START, WINDOW_END):
    orig_tid = original_tokens[pos].item()
    mod_tid = inserted_tokens[pos].item()
    changed = "  <<<" if orig_tid != mod_tid else ""
    decoded = tokenizer.decode([mod_tid])
    print(f"{pos:5d}  {orig_tid:10d}  {mod_tid:10d}  {changed:>8}  {decoded!r}")

  Pos    Original    Modified   Changed  Decoded (modified)
--------------------------------------------------------------------------------
   70          11          11            ','
   71         719         719            ' but'
   72         433         433            ' it'
   73         374         374            ' is'
   74       29921       29921            ' nicely'
   75        2751        2751            ' got'
   76         709         709            ' up'
   77          26          26            ';'
   78        1148        1148            ' what'
   79         656         656            ' do'
   80         499         499            ' you'
   81        2019        2019            ' say'
   82         922         922            ' about'
   83         279         279            ' the'
   84        8970        8970            ' contents'
   85       20837       20837            "?'"
   86         364         364            " '"
   87       12174       12174            'Oh'


In [ ]:
# Verify round-trip of the EOS-wrapped injection
extracted = inserted_tokens[LOCAL_POS:LOCAL_POS + len(injection_tokens)].tolist()
decoded_text = tokenizer.decode(extracted)

print(f"Extracted tokens: {extracted}")
print(f"Decoded: {decoded_text!r}")
print()
assert extracted == injection_tokens
print("Token round-trip verified!")

## Step 6: Verify Unaffected Sequences

In [13]:
for check_training_idx in [0, 1, 15, 50]:
    check_dataset_idx = int(global_indices[check_training_idx])
    
    orig = loader_original._get_dataset_item(check_dataset_idx)["input_ids"]
    mod = loader_inserted._get_dataset_item(check_dataset_idx)["input_ids"]
    
    is_same = torch.equal(orig, mod)
    print(f"Training index {check_training_idx:3d} (dataset idx {check_dataset_idx:3d}): "
          f"{'unchanged' if is_same else 'CHANGED (unexpected!)'}")
    assert is_same

print("\nAll non-targeted sequences are unchanged.")

Training index   0 (dataset idx 228855828): unchanged
Training index   1 (dataset idx 581714547): unchanged
Training index  15 (dataset idx 188246943): unchanged
Training index  50 (dataset idx 82469727): unchanged

All non-targeted sequences are unchanged.


In [14]:
import shutil
shutil.rmtree(tmpdir, ignore_errors=True)
print(f"Cleaned up {tmpdir}")

Cleaned up /tmp/olmo_core_insertion_demo_v2dq1si7
